In [22]:
#importing librarties
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression



In [23]:
#Generating the dataset

np.random.seed(42)

# Number of apartments
n = 500

# Generate features
size_sqft = np.random.randint(400, 1500, n)
bedrooms = np.random.randint(1, 4, n)
distance_city_km = np.random.uniform(2, 15, n)

# Generate price (roughly related to size, bedrooms, and distance)
price = (
    size_sqft * 2
    + bedrooms * 500
    - distance_city_km * 100
    + np.random.normal(0, 300, n)
)

# Create DataFrame
df = pd.DataFrame({
    "size_sqft": size_sqft,
    "bedrooms": bedrooms,
    "distance_city_km": distance_city_km,
    "price": price
})

df.head()


,size_sqft,bedrooms,distance_city_km,price
0,1260,2,12.905675,2263.792497
1,1495,2,3.620615,3912.982156
2,1444,1,13.970944,1931.519615
3,521,2,13.308653,1106.318197
4,866,2,8.744895,1977.913614


In [24]:
# Compute median price
median_price = df["price"].median()
print(median_price)
# Binary classification target
df["expensive"] = (df["price"] > median_price).astype(int)

df[["price", "expensive"]].head()


2121.673975637713


,price,expensive
0,2263.792497,1
1,3912.982156,1
2,1931.519615,0
3,1106.318197,0
4,1977.913614,0


### Why is this now a classification problem?

This is a classification problem because the target variable is no longer
a continuous price value.

We converted the price into a binary category based on the median price.
So, now the model only predicts whether an apartment is expensive or not expensive,instead of predicting the exact price.


In [25]:
# feature variables
X = df[["size_sqft", "bedrooms", "distance_city_km"]]

# target variable
y = df["expensive"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [26]:
# Train a Logistic Regression model
model = LogisticRegression()
model.fit(X_train, y_train)

LogisticRegression()

# What the model outputs?
The logistic regression model outputs a probability value between 0 and 1.

This probability represents how likely an apartment is to be expensive.
Using this probability, the model assigns a class label:
1 for expensive and 0 for not expensive.



# Why logistic regression is more suitable than linear regression here?
Logistic regression is more suitable than linear regression because this
problem is about classification, not predicting exact values.

Linear regression predicts continuous numbers, which is not ideal when
the output should only be 0 or 1. Logistic regression is designed to
handle binary outcomes and gives better results for classification tasks.


In [27]:
from sklearn.metrics import accuracy_score
# Make predictions on the test set
y_pred = model.predict(X_test)

# Compute accuracy
accuracy = accuracy_score(y_test, y_pred)
accuracy



0.86

- The accuracy of the logistic regression model on the test set is 0.86,
meaning that 86% of the apartments were classified correctly.

- This value depends on the specific train/test split and may vary when
the data is split differently.

- Accuracy alone can be misleading for apartment prices because it only shows the overall percentage of correct predictions and does not explain the types of errors the model makes.

- For example, the model may achieve high accuracy while still missing
expensive apartments or incorrectly labeling cheap apartments as expensive.
These mistakes can have different financial impacts, even if the accuracy
appears good.





In [29]:
from sklearn.metrics import confusion_matrix

# Computing confusion matrix
cm = confusion_matrix(y_test, y_pred)
cm


array([[46,  9],
       [ 5, 40]])

- The confusion matrix shows that the model makes both types of classification
errors.

- The model misses some expensive apartments, which are false negatives.
5 expensive apartments were classified as not expensive.

- The model also falsely labels some cheap apartments as expensive.
9 cheap apartments were classified as expensive.


In [31]:
from sklearn.metrics import precision_score, recall_score
# Compute precision and recall
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

precision, recall


(0.8163265306122449, 0.8888888888888888)

# Compute precision and recall
- The **precision** of the model is approximately 0.82.
- Which means that about 82% of the apartments classified as expensive
are actually expensive, while some cheap apartments are incorrectly
labeled as expensive.

- The **recall** of the model is approximately 0.89.
- This means that about 89% of the truly expensive apartments are correctly
identified by the model, while a small number of expensive apartments
are missed.



# Explain what each metric means in the apartment context:-
Precision measures how reliable the model is when it predicts that an
apartment is expensive. A high precision means that most apartments
labeled as expensive are truly expensive.

Recall measures how well the model identifies all expensive apartments.
A high recall means that most expensive apartments are correctly detected
and fewer valuable apartments are missed.


# Which metric is more important for a real estate investor? Why?
Recall is more important for a real estate investor because it measures
how many expensive apartments are correctly identified.

Missing an expensive apartment can result in a lost investment
opportunity. It is usually better to review some extra apartments than
to miss valuable properties.


In [33]:
# Get probabilities for the positive class (expensive = 1)
y_prob = model.predict_proba(X_test)[:, 1]

# Apply a higher decision threshold
threshold = 0.7
y_pred_07 = (y_prob >= threshold).astype(int)



In [34]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score

cm_07 = confusion_matrix(y_test, y_pred_07)
precision_07 = precision_score(y_test, y_pred_07)
recall_07 = recall_score(y_test, y_pred_07)

cm_07, precision_07, recall_07


(array([[52,  3],
        [10, 35]]),
 0.9210526315789473,
 0.7777777777777778)

### Change the classification threshold (e.g., from 0.5 to 0.7 or 0.3)
When the decision threshold is increased from 0.5 to 0.7, the model becomes
more strict when predicting an apartment as expensive.

As a result, precision increases to about 0.92, meaning that most apartments predicted as expensive are truly expensive.

At the same time, recall decreases to about 0.78, meaning that the model
misses more expensive apartments than before.



## 7. Conceptual Questions ()

**Q: What does overfitting mean in classification?**  
Overfitting in classification occurs when a model learns the training data
too well, including noise and irrelevant patterns. Due to which, the model
performs very well on the training data but does not generalize well to new
or unseen data.

**Q: Why can a single train/test split be misleading?**  
- A single train/test split can be misleading because the results depend on
which data points are selected for testing.
- If the test data happens to be easier or harder, the model’s performance
may look better or worse than it actually is.


**Q: How does cross-validation help?**  
Cross-validation helps by training and testing the model on multiple
different splits of the data. Using multiple train/test splits does not increase the model’s accuracy.Instead, it provides a more reliable estimate of how the model performs on different subsets of data and reduces the impact of random variation in the data split.

**Q: What problem does regularization solve?**  
Regularization solves the problem of overfitting. As it prevents the model from becoming too complex by limiting large feature, weights, helping the model generalize better to new and unseen data.

